# Medical Assistant — A RAG-Based Clinical Knowledge Assistant built on the Merck Manual

**Program:** Post Graduate Program in AI & ML in Business Applications — McCombs School of Business, The University of Texas at Austin

**Module:** Natural Language Processing with Generative AI

**Project track:** Full-code

**Submitted by:** Anoop Krishnan

---

**Solution roadmap**

| # | Stage | What is built |
|---|-------|---------------|
| 1 | Question answering with a raw LLM | Mistral-7B-Instruct (GGUF, 4-bit) answers the five clinical questions from parametric memory only |
| 2 | Prompt engineering & parameter tuning | Five prompt / decoding combinations are applied to every question and compared |
| 3 | Data preparation for RAG | The 4,114-page Merck Manual is loaded, cleaned, chunked, embedded and indexed in ChromaDB |
| 4 | Question answering with RAG | The same questions are answered with retrieved manual passages; six chunking / retriever / LLM configurations are compared |
| 5 | Output evaluation | An LLM-as-a-judge scores every RAG answer for **groundedness** and **relevance** |
| 6 | Insights & recommendations | What the prototype proves, where it fails, and how a hospital should deploy it |

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.4 MB ? eta -:--:--     ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 5.1/9.4 MB 152.8 MB/s eta 0:00:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 148.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 260.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 307.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 246.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have hu

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.
- If the source build of `llama-cpp-python` fails on a newer Colab CUDA toolchain, the pre-built CUDA wheel can be used instead: `!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q` (the rest of the notebook is unchanged).
- The runtime for this notebook is a **Google Colab T4 GPU** (Runtime → Change runtime type → T4 GPU). `n_gpu_layers=-1` below offloads every transformer layer of the 7B model to the GPU; on CPU the same code runs but generation is roughly 20x slower.

In [2]:
# For installing the libraries & downloading models from HF Hub.
# Note: pandas / numpy are left at the versions Colab ships, because the pinned pandas==2.2.2 has no pre-built wheel for
# Colab's current Python version and fails to compile from source. Everything else follows the template's pins.
!pip install huggingface_hub==0.35.3 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 "langchain-text-splitters>=0.3,<0.4" chromadb==1.1.1 sentence-transformers==5.1.1 "pandas>=2.2" "numpy>=2.0" -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# ---- Housekeeping: keep the notebook output clean ---------------------------------
import warnings, logging, os
warnings.filterwarnings("ignore")                       # silence LangChain / HF deprecation notices
logging.getLogger("chromadb").setLevel(logging.ERROR)    # silence Chroma telemetry chatter
os.environ["ANONYMIZED_TELEMETRY"] = "False"             # Chroma: do not phone home
os.environ["TOKENIZERS_PARALLELISM"] = "false"           # HF tokenizers: avoid fork warnings

# ---- Libraries for processing dataframes, text -------------------------------------
import json, re, time, textwrap, shutil
import tiktoken
import pandas as pd
import numpy as np

# ---- Libraries for loading data, chunking, embedding and vector databases ----------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# ---- Libraries for downloading and loading the LLM ---------------------------------
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# LangChain re-enables its own deprecation category when it is imported, so the filter is re-applied after the imports
warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 200)
print("Libraries imported successfully.")

Libraries imported successfully.


## Question Answering using LLM

**Why start with a plain LLM?** Before building retrieval, we need a *baseline*: how well does a general-purpose
open-weights model answer specialist clinical questions purely from what it memorised during pre-training? Every
improvement later in the notebook (prompting, then RAG) is measured against this baseline.

**Model choice — Mistral-7B-Instruct-v0.2 (GGUF, Q4_K_M quantisation)**

| Consideration | Rationale |
|---|---|
| Open weights, Apache-2.0 | No API cost, no patient data leaves the hospital's infrastructure — a hard requirement in healthcare |
| 7B parameters | Strong instruction following for its size; fits comfortably on a 16 GB T4 when 4-bit quantised (about 4.4 GB) |
| `Q4_K_M` | The best accuracy/size trade-off in the K-quant family; negligible quality loss vs. FP16 for QA tasks |
| `llama-cpp-python` | Runs GGUF files with CUDA offload (`n_gpu_layers=-1`) and exposes every sampling parameter we need to tune |

#### Downloading and Loading the model

In [4]:
# Download the quantised GGUF weights from the Hugging Face Hub (cached after the first run, ~4.4 GB)
MODEL_REPO = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_FILE = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print("Model weights at:", model_path)

Model weights at: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf


In [5]:
# Load the model with llama-cpp-python
#   n_ctx        : context window in tokens. 4096 leaves room for a system prompt + several retrieved chunks + the answer
#   n_gpu_layers : -1 offloads ALL layers to the GPU (T4). Set to 0 to run on CPU.
#   n_batch      : prompt tokens processed per batch; 512 is a good default for a T4
#   verbose      : suppress llama.cpp's per-call performance logs to keep the output readable
CONTEXT_WINDOW = 4096

# Compatibility fix: with verbose=False, llama-cpp-python silences llama.cpp's C-level logs by calling sys.stdout.fileno(),
# which the Colab (ipykernel on Python 3.13) stdout stream does not support and raises io.UnsupportedOperation.
# The helper below does the same job at the file-descriptor level (fd 1 / fd 2), so the notebook output stays clean.
import llama_cpp._utils as _llama_utils

def _quiet_enter(self):
    if self.disable:
        return self
    self._devnull = os.open(os.devnull, os.O_WRONLY)
    self._saved = (os.dup(1), os.dup(2))
    os.dup2(self._devnull, 1)
    os.dup2(self._devnull, 2)
    return self

def _quiet_exit(self, *_):
    if self.disable:
        return
    os.dup2(self._saved[0], 1)
    os.dup2(self._saved[1], 2)
    for fd in (*self._saved, self._devnull):
        os.close(fd)

_llama_utils.suppress_stdout_stderr.__enter__ = _quiet_enter
_llama_utils.suppress_stdout_stderr.__exit__ = _quiet_exit

llm = Llama(
    model_path=model_path,
    n_ctx=CONTEXT_WINDOW,
    n_gpu_layers=-1,
    n_batch=512,
    verbose=False,
)
print("Model loaded. Context window:", CONTEXT_WINDOW, "tokens")

Model loaded. Context window: 4096 tokens


**Observation — model loading.** The 4-bit model loads in about ten seconds from the local cache and occupies
roughly 4.5 GB of the T4's 16 GB, leaving ample headroom for a 4,096-token context. A context window this size
matters for the RAG stage: with `k = 5` retrieved chunks of about 1,000 characters (roughly 250 tokens each) plus the
system prompt, the prompt alone can approach 1,500 tokens, which the template's default of 512 would overflow.

#### Response

Mistral-Instruct was fine-tuned on conversations wrapped in `[INST] ... [/INST]` tags. Sending the raw question without
those tags makes the model *continue* the text rather than *answer* it, so the helper below applies the chat template
and exposes the decoding parameters we will tune later:

* `max_tokens` — the maximum length of the generated answer;
* `temperature` — 0 makes decoding greedy (deterministic); higher values sample more diverse tokens;
* `top_p` / `top_k` — nucleus / top-k truncation of the sampling distribution;
* `repeat_penalty` — discourages the model from looping on the same phrase.

In [6]:
def build_prompt(query, system_message=None):
    """Wrap a query (and an optional system message) in the Mistral-Instruct chat template."""
    if system_message:
        return f"[INST] {system_message.strip()}\n\n{query.strip()} [/INST]"
    return f"[INST] {query.strip()} [/INST]"


def response(query, max_tokens=128, temperature=0, top_p=0.95, top_k=50,
             repeat_penalty=1.1, system_message=None):
    """Generate an answer from the LLM alone (no retrieval)."""
    model_output = llm(
        prompt=build_prompt(query, system_message),
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["[INST]", "</s>"],
    )
    return model_output["choices"][0]["text"].strip()


def show(text, width=110):
    """Pretty-print a model answer with wrapped lines so it reads well in the exported HTML."""
    for paragraph in text.split("\n"):
        print(textwrap.fill(paragraph, width=width) if paragraph.strip() else "")


def timed_response(query, **kwargs):
    """Generate, print and time a baseline answer."""
    start = time.time()
    answer = response(query, **kwargs)
    show(answer)
    print(f"\n[{len(answer.split())} words | {time.time() - start:.1f} s]")
    return answer

The five clinical questions from the problem statement are stored once so that every stage of the notebook answers
exactly the same set. The baseline uses greedy decoding (`temperature = 0`) and `max_tokens = 256`, enough for a
complete clinical answer while keeping the comparison with later stages fair.

In [7]:
questions = {
    "Q1": "What is the protocol for managing sepsis in a critical care unit?",
    "Q2": "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Q3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Q4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "Q5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
}

# Every stage stores its answers here so that they can be compared at the end
results = {"LLM baseline": {}, "Prompt engineering": {}, "RAG": {}}
BASELINE_MAX_TOKENS = 256

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [8]:
results["LLM baseline"]["Q1"] = timed_response(questions["Q1"], max_tokens=BASELINE_MAX_TOKENS)

Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it
effectively in a critical care unit requires a multidisciplinary approach. Here are some key steps in the
protocol for managing sepsis in a critical care unit:

1. Early recognition and diagnosis: Identify sepsis early by recognizing the signs and symptoms, such as
fever, chills, rapid heart rate, rapid breathing, confusion, and low blood pressure. Use the Sequential Organ
Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) to help identify patients
at risk of developing sepsis.
2. Resuscitation: Provide early and adequate fluid resuscitation to maintain adequate tissue perfusion and
organ function. Use intravenous fluids, such as normal saline or lactated Ringer's solution, to restore
intravascular volume. Monitor hemodynamic parameters, such as mean arterial pressure (MAP), central venous
pressure (CVP), and cardiac output, to guide fluid resuscitatio

**Observation — Q1 (sepsis).** The unaided model produces a plausible, textbook-style protocol — early recognition
with SOFA/qSOFA, crystalloid resuscitation guided by MAP, CVP and cardiac output, then antibiotics — and is cut off
mid-sentence by the 256-token budget before it reaches source control or vasopressors. Everything it says is
generic sepsis knowledge from pre-training: nothing is attributable to the Merck Manual, so a clinician cannot
check a single threshold against the reference the hospital actually uses.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [9]:
results["LLM baseline"]["Q2"] = timed_response(questions["Q2"], max_tokens=BASELINE_MAX_TOKENS)

Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ
located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Abdominal pain: The pain usually starts as a mild discomfort around the navel and then moves to the lower
right side of the abdomen. The pain may be constant or intermittent, and it may worsen with movement or
pressure on the abdomen.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal discomfort.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in advanced cases.
4. Fever: A fever may be present, particularly if the appendix has ruptured.
5. Diarrhea or constipation: Some people with appendicitis experience diarrhea, while others have
constipation.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: Passing gas can be difficult due to pressure on the append

**Observation — Q2 (appendicitis).** The three-part question exposes the baseline's biggest weakness: the model
spends the entire budget enumerating seven symptoms (some of them, such as diarrhoea and abdominal swelling, are
soft signs) and is truncated before it ever answers "can it be cured via medicine?" or names appendectomy. A
clinician asking this question would receive no treatment guidance at all. Answer length and structure clearly
have to be controlled.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [10]:
results["LLM baseline"]["Q3"] = timed_response(questions["Q3"], max_tokens=BASELINE_MAX_TOKENS)

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in
small patches on the scalp or other parts of the body. The exact cause of alopecia areata is not known, but
it's believed to be related to an abnormal immune response that attacks the hair follicles. Here are some
effective treatments and possible causes for this condition:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system, allowing
the hair to grow back. They can be applied topically or taken orally.
2. Immunotherapies: Other immunotherapies like minoxidil, anthralin, and contact sensitizers have also been
used to treat alopecia areata. These treatments work by stimulating the immune system to promote hair growth.
3. Diet and lifestyle changes: Certain nutritional deficiencies or stress can trigger hair loss. Eating a
healthy diet rich in vitamins, minerals, and antioxidants may help improve hair health. Reducing stress
through re

**Observation — Q3 (patchy hair loss).** The question deliberately *describes* the condition instead of naming it,
and the model correctly infers alopecia areata and its autoimmune mechanism. The treatment list, however, mixes
correct options (topical or oral corticosteroids) with a mislabelled one — minoxidil and anthralin are called
"immunotherapies" — and drifts into lifestyle advice (diet, yoga, meditation). Fluent, mostly right, and with no
way to tell which parts are reliable.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [11]:
results["LLM baseline"]["Q4"] = timed_response(questions["Q4"], max_tokens=BASELINE_MAX_TOKENS)

Treatment for a physical injury to the brain tissue depends on the severity and location of the injury. Here
are some common treatments that may be recommended:

1. Emergency care: If the injury is recent, the person may require emergency care to address any life-
threatening conditions such as airway obstruction, breathing difficulties, or excessive bleeding.
2. Surgery: Depending on the location and severity of the injury, surgery may be necessary to remove hematomas
(clots of blood) or repair skull fractures.
3. Medications: Various medications may be prescribed to manage symptoms such as pain, swelling, seizures, or
infections.
4. Rehabilitation: Rehabilitation programs can help individuals regain lost skills and improve their overall
functioning. This may include physical therapy, occupational therapy, speech therapy, and cognitive
rehabilitation.
5. Assistive devices: Depending on the extent of the impairment, assistive devices such as wheelchairs,
walkers, or communication aids 

**Observation — Q4 (traumatic brain injury).** The description avoids the clinical term and the model answers at
the level of a patient leaflet: emergency care, surgery for haematomas, "various medications", rehabilitation,
assistive devices and support groups. There is no acute-care pathway — no mention of intracranial pressure,
oxygenation targets or when surgical evacuation is indicated — and the answer is again truncated. Generic guidance
is exactly what a critical-care team does *not* need.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [12]:
results["LLM baseline"]["Q5"] = timed_response(questions["Q5"], max_tokens=BASELINE_MAX_TOKENS)

If a person has fractured their leg during a hiking trip, it is important that they receive proper medical
attention as soon as possible. Here are some necessary precautions and treatment steps:

1. Assess the situation: Check the severity of the injury and assess the overall condition of the person. If
the fracture is open or the person is in severe pain, distress, or unable to move, do not attempt to move them
without professional help. Call for emergency medical assistance immediately.

2. Provide first aid: If the person's condition allows, provide basic first aid such as cleaning and dressing
any wounds, applying a sterile bandage or splint, and administering pain relief medication if available. Do
not attempt to realign the bone or apply excessive pressure to the fracture.

3. Immobilize the leg: Use a makeshift splint or sling to immobilize the leg as best as possible to prevent
further injury or movement of the fractured bone. This will help reduce pain and swelling, and preven

**Observation — Q5 (leg fracture on a hike).** The baseline gives sensible lay first aid (assess, do not move an
open fracture, improvise a splint, arrange transport) but stops mid-sentence at the 256-token limit before any
definitive care or recovery guidance. The audience is also wrong for a clinical decision-support tool: it reads
as advice to a bystander rather than to a clinician.

### Summary — question answering with the LLM alone

Across the five questions the same pattern appears. The answers are fluent and broadly sensible, but they are
**unverifiable** (no source anywhere), **incomplete** (four of the five were truncated by the 256-token budget, and
Q2 never reached the treatment part of the question), **uneven in depth** (strong on textbook sepsis, leaflet-level
on brain injury), and **occasionally wrong in detail** (minoxidil described as immunotherapy). Fluency without
provenance is the core risk of deploying a raw LLM in healthcare — the next two sections address it, first by
controlling the model through prompting, then by grounding it in the Merck Manual.

## Question Answering using LLM with Prompt Engineering

Prompt engineering changes *how we ask*, parameter tuning changes *how the model samples*. Five combinations are
defined below and every one is applied to all five questions, so their effect can be compared question by question.

| ID | Prompting technique | Decoding parameters | What it tests |
|---|---|---|---|
| **PE-1** | Role prompt — clinical decision-support assistant, answer for physicians | `temperature 0`, `max_tokens 400` | Does a persona + audience raise clinical precision? |
| **PE-2** | Same role prompt | `temperature 0.7`, `top_p 0.9`, `top_k 40` | How much does stochastic sampling change a medical answer? |
| **PE-3** | Structured-output prompt — fixed section headings | `temperature 0.2`, `top_k 20`, `max_tokens 450` | Does a mandated structure make multi-part questions complete? |
| **PE-4** | Chain-of-thought — reason step by step before answering | `temperature 0.3`, `max_tokens 512` | Does explicit reasoning improve inference when the condition is only described? |
| **PE-5** | Constrained prompt — concise bullets, admit uncertainty, no invented drugs/doses | `temperature 0`, `max_tokens 250`, `repeat_penalty 1.2` | Can we get a safe, short bedside answer? |

In [13]:
SYSTEM_ROLE = """You are a clinical decision-support assistant used by physicians in a hospital.
Answer medical questions accurately, using standard medical terminology, and prioritise information
that helps a clinician act. Do not include disclaimers about consulting a doctor - the user is a doctor."""

SYSTEM_STRUCTURED = """You are a clinical decision-support assistant used by physicians in a hospital.
Answer using exactly these section headings, each followed by 2-5 concise sentences or bullet points:
1. Condition / likely diagnosis
2. Key signs and symptoms
3. Recommended management (medical and, if applicable, surgical)
4. Precautions, follow-up and recovery
If a section does not apply to the question, write 'Not applicable'."""

SYSTEM_COT = """You are a clinical decision-support assistant used by physicians in a hospital.
First, reason step by step: identify the condition being described, then recall its causes, then its evidence-based
management. Write this reasoning under the heading 'Reasoning'. Then write the final recommendation under the
heading 'Answer', in no more than 8 sentences."""

SYSTEM_CONSTRAINED = """You are a clinical decision-support assistant used by physicians in a hospital.
Rules:
- Answer in at most 6 short bullet points.
- Use only well-established medical knowledge. Do not invent drug names, doses or statistics.
- If you are not certain about a detail, say 'uncertain' rather than guessing.
- Do not repeat yourself."""

prompt_configs = {
    "PE-1  Role prompt, greedy":              dict(system_message=SYSTEM_ROLE,        max_tokens=400, temperature=0.0, top_p=0.95, top_k=50, repeat_penalty=1.1),
    "PE-2  Role prompt, high temperature":    dict(system_message=SYSTEM_ROLE,        max_tokens=400, temperature=0.7, top_p=0.90, top_k=40, repeat_penalty=1.1),
    "PE-3  Structured output, low temp":      dict(system_message=SYSTEM_STRUCTURED,  max_tokens=450, temperature=0.2, top_p=0.95, top_k=20, repeat_penalty=1.1),
    "PE-4  Chain-of-thought":                 dict(system_message=SYSTEM_COT,         max_tokens=512, temperature=0.3, top_p=0.95, top_k=50, repeat_penalty=1.1),
    "PE-5  Constrained bullets, greedy":      dict(system_message=SYSTEM_CONSTRAINED, max_tokens=250, temperature=0.0, top_p=0.95, top_k=50, repeat_penalty=1.2),
}


def run_prompt_configs(qid):
    """Apply every prompt/parameter combination to one question and print the answers."""
    query = questions[qid]
    results["Prompt engineering"][qid] = {}
    for name, cfg in prompt_configs.items():
        print("=" * 110)
        print(f"{name}   |   temperature={cfg['temperature']}  top_p={cfg['top_p']}  top_k={cfg['top_k']}  max_tokens={cfg['max_tokens']}  repeat_penalty={cfg['repeat_penalty']}")
        print("=" * 110)
        start = time.time()
        answer = response(query, **cfg)
        show(answer)
        print(f"\n[{len(answer.split())} words | {time.time() - start:.1f} s]\n")
        results["Prompt engineering"][qid][name] = answer

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [14]:
run_prompt_configs("Q1")

PE-1  Role prompt, greedy   |   temperature=0.0  top_p=0.95  top_k=50  max_tokens=400  repeat_penalty=1.1
Sepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care
unit, managing sepsis involves the following steps:

1. Early recognition and suspicion: Identify patients at risk based on clinical signs, laboratory findings,
and epidemiological factors.
2. Rapid administration of antibiotics: Start broad-spectrum antibiotics as soon as possible based on
suspected infection source and local microbiology data.
3. Fluid resuscitation: Aim for adequate tissue perfusion by administering crystalloids or colloids to
maintain mean arterial pressure (MAP) ≥65 mmHg and central venous oxygen saturation (ScvO2) >70%.
4. Vasopressor support: If MAP remains below target despite fluid resuscitation, initiate vasopressors to
maintain adequate tissue perfusion.
5. Corticosteroids: Consider administering corticosteroids in cases of septic shock that do not r

**Observation — Q1 with prompt engineering.** The role prompt (PE-1) transforms the register: the answer now reads
like an ICU handover, with a ten-step pathway and explicit targets (MAP ≥ 65 mmHg, ScvO2 > 70 %) that the baseline
never gave. PE-2's higher temperature keeps the substance but reorders and rewords it — acceptable for
brainstorming, unacceptable for a protocol, where two clinicians asking the same question should receive the same
answer. PE-3's headings cleanly separate signs from management. PE-4 spends most of its budget on the "Reasoning"
section and then restates the same eight points in its "Answer", so the extra length buys no extra content. PE-5 is the shortest usable
bedside summary, but despite the instruction not to invent doses it states a hydrocortisone dose from memory —
a prompt rule cannot make the model *know* what it does not know. None of the five can attribute a single
threshold to a reference.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
run_prompt_configs("Q2")

PE-1  Role prompt, greedy   |   temperature=0.0  top_p=0.95  top_k=50  max_tokens=400  repeat_penalty=1.1
Appendicitis is a common inflammatory condition characterized by the following symptoms:

1. Periumbilical or right lower quadrant abdominal pain that begins as a mild discomfort and progresses to
constant and severe pain.
2. Anorexia (loss of appetite) and nausea, which may lead to vomiting.
3. Pyrexia (fever), often with a temperature above 37.8°C (100°F).
4. Localized tenderness in the right lower quadrant of the abdomen.
5. Rebound tenderness and guarding, which may indicate peritonitis.
6. Constipation or diarrhea.

Appendicitis cannot be cured via medicine alone due to the risk of rupture and potential complications. Once
diagnosed, an appendectomy is the standard surgical procedure for treating appendicitis. This involves
removing the inflamed appendix through either open surgery or laparoscopic methods. The goal is to prevent
rupture and subsequent peritonitis, which can le

**Observation — Q2 with prompt engineering.** Every configuration now answers all three parts of the question —
symptoms, "no, medicine alone does not cure it", and open or laparoscopic appendectomy — where the baseline never
reached the treatment at all. The larger token budget is the main reason; the structured prompt (PE-3) and the
bullet constraint (PE-5) additionally guarantee that each part gets its own heading or bullet. Two small regressions
are instructive. PE-3 appends a stray "(Not applicable)" before its last sentence — a rigid template can be
over-applied. PE-4's final "Answer" opens with "based on the symptoms described, it appears that the patient may be
suffering from appendicitis", re-diagnosing a case that was never presented and urging "prompt medical attention" as
if addressing a lay reader — chain-of-thought lengthens the generation and lets the framing drift.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
run_prompt_configs("Q3")

PE-1  Role prompt, greedy   |   temperature=0.0  top_p=0.95  top_k=50  max_tokens=400  repeat_penalty=1.1
I. Possible Causes of Sudden Patchy Hair Loss:

A. Alopecia Areata: An autoimmune disorder that results in the sudden loss of hair from specific areas of the
scalp or body. It often presents as round, smooth patches of baldness.
B. Telogen Effluvium: A condition where a large number of hairs enter the resting phase (telogen) at once,
leading to increased shedding and visible thinning.
C. Traction Alopecia: Hair loss caused by pulling or tension on the hair, often due to hairstyles that are too
tight or frequent manipulation.
D. Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, or vitamin B12 can lead to hair
loss.
E. Stress: Emotional or physical stress can cause temporary hair loss.
F. Medications: Certain medications such as chemotherapy drugs, antidepressants, and beta-blockers have been
known to cause hair loss.
G. Hormonal Imbalances: Thyroid disorders, po

**Observation — Q3 with prompt engineering.** Because the condition is only described, the prompts change *what*
the model decides to answer. PE-1 and PE-3 respond with a broad differential (alopecia areata, telogen effluvium,
traction alopecia and, in PE-1, nutritional, hormonal and infectious causes) rather than committing to the most
likely diagnosis. PE-2 and the chain-of-thought prompt (PE-4) do commit to alopecia areata and then treat it —
PE-4's explicit "Reasoning" is the clinically useful behaviour here, because making the inference visible lets a
clinician check it — but both pad the list with hair transplantation and dietary supplements, and PE-4's "Answer"
addresses the reader as the patient ("it appears that you are experiencing ... consult your healthcare provider")
despite the physician persona. The minoxidil-as-immunomodulator error from the baseline recurs in four of the five
answers: prompting does not correct a fact the model has learned wrongly.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
run_prompt_configs("Q4")

PE-1  Role prompt, greedy   |   temperature=0.0  top_p=0.95  top_k=50  max_tokens=400  repeat_penalty=1.1
For a person with a traumatic brain injury (TBI), treatment goals include addressing acute injuries,
preventing complications, and promoting recovery. Recommended treatments may include:

1. Initial evaluation and stabilization: This includes assessing the severity of the injury, managing airway,
breathing, and circulation, controlling intracranial pressure, and providing adequate hydration and nutrition.

2. Surgical intervention: Depending on the type and location of the brain injury, surgery may be required to
remove hematomas or decompress swelling.

3. Medications: Various medications may be used to manage symptoms such as seizures, pain, and increased
intracranial pressure.

4. Rehabilitation: Physical, occupational, speech, and cognitive therapies can help improve function and
independence in daily activities.

5. Supportive care: This includes providing adequate nutrition, 

**Observation — Q4 with prompt engineering.** The role and structured prompts move the answer from leaflet level
toward an acute-care pathway (airway, breathing, circulation, intracranial-pressure control, craniotomy or
decompressive craniectomy, rehabilitation), and PE-2 names the medication classes involved (anticonvulsants,
diuretics, sedatives, analgesics). The constrained prompt (PE-5), however, lists "steroids for
reducing inflammation and swelling in the injured area" — advice that current trauma guidance does not support for
traumatic brain injury — and nothing in the answer signals that this is unsafe; the appended "(Uncertain about
specific medications or dosages)" is the behaviour the prompt asked for, but it is attached to the wrong claim.
This question is the clearest demonstration that a well-prompted LLM can be confidently wrong on a treatment
decision, and why grounding in a vetted reference is not optional.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [18]:
run_prompt_configs("Q5")

PE-1  Role prompt, greedy   |   temperature=0.0  top_p=0.95  top_k=50  max_tokens=400  repeat_penalty=1.1
A leg fracture sustained during a hiking trip requires prompt medical attention. Here are the necessary
precautions and treatment steps:

1. Immobilization: Apply a splint or use a fiberglass cast to immobilize the affected leg from the injury site
down to the toe. This will help prevent further damage and promote healing.

2. Pain management: Administer pain medication as prescribed by a healthcare professional to manage discomfort
and improve patient comfort.

3. Assess for open fractures: Inspect the wound for signs of an open fracture, such as visible bone or tissue
damage. If present, cover the wound with a sterile dressing and apply pressure to control bleeding.

4. Monitor vital signs: Keep track of the patient's pulse, respiratory rate, blood pressure, and temperature.
This will help ensure that the patient remains stable during transport and treatment.

5. Transport: Arran

**Observation — Q5 with prompt engineering.** The role prompt (PE-1) finally covers the whole arc — immobilisation,
open-fracture check, vital signs, transport, tetanus status, orthopaedic follow-up and rehabilitation — that the
baseline truncated. PE-2 puts airway, breathing and circulation first and reaches definitive care (pins, plates,
cast or traction) but is cut off at follow-up. PE-3 gives the most concise complete answer, correctly separating
field management (RICE, immobilisation) from surgery, infection prevention and rehabilitation. PE-4 is again long
(303 words) and its "Answer" repeats the "Reasoning" almost point for point. PE-5 is the format a triage nurse or
paramedic would actually want: ABC first, immobilise, analgesia, monitor, rehabilitate, refer.

### Summary — what prompt engineering and parameter tuning did (and did not) fix

* **Budget and structure fixed completeness.** Raising `max_tokens` and mandating sections (PE-3) or bullets (PE-5)
  turned truncated, single-part answers into complete multi-part ones — the largest quality jump in this section.
* **Deterministic decoding is non-negotiable for clinical protocols.** PE-2 (temperature 0.7) gives a differently
  ordered and worded answer every time the same question is asked; in a protocol setting two clinicians must
  receive the same answer.
* **Chain-of-thought helps when the condition must be inferred** (Q3) by making the inference visible, at the cost
  of the longest and most repetitive answers and, in Q2 and Q3, a drift toward a patient-facing voice.
* **Constraints reduce padding but not ignorance.** PE-5 stopped the model from looping and asked it to flag
  uncertainty, yet it still produced a dose from memory (Q1) and a treatment that trauma guidance does not
  support (Q4). A prompt cannot add knowledge the model lacks, cannot correct a fact it has learned wrongly, and
  cannot make it cite the manual.

The remaining gap — provenance and verifiable, institution-approved content — is exactly what retrieval-augmented
generation addresses, so the next section builds the knowledge base.

## Data Preparation for RAG

**RAG in one sentence:** instead of asking the model to *remember* the Merck Manual, we *show* it the relevant pages at
question time. The pipeline is: load the PDF → clean the page text → split it into overlapping chunks → embed each chunk
as a vector → store the vectors in ChromaDB → retrieve the chunks most similar to a question → pass them to the LLM as
context.

### Loading the Data

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Path to the Merck Manual PDF in Google Drive (update if the file is stored elsewhere)
PDF_PATH = "/content/drive/MyDrive/medical_diagnosis_manual.pdf"

start = time.time()
loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()          # one LangChain Document per PDF page, with page number in metadata
print(f"Loaded {len(pages):,} pages in {time.time() - start:.1f} s")

Loaded 4,114 pages in 14.9 s


### Data Overview

#### Checking the first 5 pages

In [21]:
for i, page in enumerate(pages[:5]):
    print(f"--- Page index {i} (PDF page {page.metadata['page'] + 1}) | {len(page.page_content):,} characters ---")
    print(page.page_content[:600].strip())
    print()

--- Page index 0 (PDF page 1) | 172 characters ---
adkanoop@gmail.com
PFAQMR1H36
This file is meant for personal use by adkanoop@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page index 1 (PDF page 2) | 172 characters ---
adkanoop@gmail.com
PFAQMR1H36
This file is meant for personal use by adkanoop@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page index 2 (PDF page 3) | 8,936 characters ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ..............................

**Observation — first pages.** The first two pages contain nothing but a per-copy licence watermark
("*This file is meant for personal use by … only*"), and the same three-line watermark is stamped on **every** page of
the manual. Page 3 onward is the table of contents. If the watermark were left in place it would be embedded into every
chunk, waste roughly 170 characters of each chunk's budget, and — because identical text appears in all vectors —
slightly blur the similarity search. It is therefore removed before chunking.

#### Checking the number of pages

In [22]:
print("Number of pages in the manual :", len(pages))

# Size of the corpus in characters and (approximate) tokens, using the same BPE tokeniser family as most LLMs
encoder = tiktoken.get_encoding("cl100k_base")
total_chars = sum(len(p.page_content) for p in pages)
total_tokens = sum(len(encoder.encode(p.page_content)) for p in pages)
print(f"Total characters              : {total_chars:,}")
print(f"Approximate tokens            : {total_tokens:,}")
print(f"Average characters per page   : {total_chars / len(pages):,.0f}")
print(f"Pages needed to fill a {CONTEXT_WINDOW:,}-token context window: ~{CONTEXT_WINDOW / (total_tokens / len(pages)):.0f}")

Number of pages in the manual : 4114
Total characters              : 13,654,294
Approximate tokens            : 3,256,238
Average characters per page   : 3,319
Pages needed to fill a 4,096-token context window: ~5


**Observation — corpus size.** At about 3.3 million tokens the manual is roughly **800 times larger** than the
model's 4,096-token context window; only about five average pages fit at a time. The manual can never be passed to
the LLM whole, which is precisely why retrieval must select the right pages for each question.

In [23]:
# ---- Cleaning: strip the licence watermark and normalise whitespace ---------------------
WATERMARK_PATTERNS = [
    r"adkanoop@gmail\.com",
    r"PFAQMR1H36",
    r"This file is meant for personal use by .*? only\.",
    r"Sharing or publishing the contents in part or full is liable for legal action\.",
]

def clean_text(text):
    for pattern in WATERMARK_PATTERNS:
        text = re.sub(pattern, "", text)
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces
    text = re.sub(r"\n{3,}", "\n\n", text)      # collapse blank lines
    return text.strip()

cleaned_pages = []
for p in pages:
    text = clean_text(p.page_content)
    if len(text) < 50:                          # drop pages that were only a watermark
        continue
    cleaned_pages.append(Document(page_content=text, metadata={"page": p.metadata["page"] + 1, "source": "Merck Manual 19th ed."}))

print(f"Pages retained after cleaning : {len(cleaned_pages):,}  (dropped {len(pages) - len(cleaned_pages)} watermark-only pages)")
print(f"Characters removed            : {total_chars - sum(len(p.page_content) for p in cleaned_pages):,}")
print("\nSample cleaned page (PDF page 2454 - opening of Chapter 227, Sepsis & Septic Shock):\n")
print(next(p for p in cleaned_pages if p.metadata["page"] == 2454).page_content[:900])

Pages retained after cleaning : 4,112  (dropped 2 watermark-only pages)
Characters removed            : 713,110

Sample cleaned page (PDF page 2454 - opening of Chapter 227, Sepsis & Septic Shock):

Chapter 227. Sepsis and Septic Shock
Introduction
(See also Ch. 226.)
Sepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic
response to bacterial infection. In severe sepsis and septic shock, there is critical reduction in
tissue perfusion. Common causes include gram-negative organisms, staphylococci, and
meningococci. Symptoms often begin with shaking chills and include fever, hypotension,
oliguria, and confusion. Acute failure of multiple organs, including the lungs, kidneys, and liver,
can occur. Treatment is aggressive fluid resuscitation, antibiotics, surgical excision of infected
or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of
blood glucose and administration of corticosteroids and activated protein C.


### Data Chunking

**Why chunk, and why these settings?** Embedding models have a short input limit (256 word-pieces for MiniLM) and a
vector that averages a whole page dilutes the specific paragraph a question needs. `RecursiveCharacterTextSplitter`
splits on paragraph → line → sentence → word boundaries in that order, so chunks respect the manual's natural
structure. The starting configuration is **1,000 characters with 200-character overlap** (roughly 250 tokens); the overlap
ensures a sentence cut at a boundary survives intact in at least one chunk. A smaller chunk size is compared in the
fine-tuning section.

In [24]:
def chunk_documents(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],   # paragraph -> line -> sentence -> word -> character
        length_function=len,
    )
    return splitter.split_documents(docs)

CHUNK_SIZE, CHUNK_OVERLAP = 1000, 200
chunks = chunk_documents(cleaned_pages, CHUNK_SIZE, CHUNK_OVERLAP)

chunk_lengths = [len(c.page_content) for c in chunks]
print(f"Chunk size / overlap        : {CHUNK_SIZE} / {CHUNK_OVERLAP} characters")
print(f"Number of chunks            : {len(chunks):,}")
print(f"Average chunks per page     : {len(chunks) / len(cleaned_pages):.1f}")
print(f"Chunk length (chars) - mean : {np.mean(chunk_lengths):.0f}, median : {np.median(chunk_lengths):.0f}, min : {min(chunk_lengths)}, max : {max(chunk_lengths)}")
print("\nExample chunk:\n")
print(chunks[len(chunks) // 2].page_content)
print("\nmetadata:", chunks[len(chunks) // 2].metadata)

Chunk size / overlap        : 1000 / 200 characters
Number of chunks            : 17,047
Average chunks per page     : 4.1
Chunk length (chars) - mean : 868, median : 943, min : 61, max : 1000

Example chunk:

Treatment
• Supportive care
• Riluzole for bulbar-variant ALS
There is no specific treatment. However, an antiglutamate drug, riluzole 50 mg po bid, prolongs life in
patients with progressive bulbar palsy. A multidisciplinary team approach helps patients cope with
progressive neurologic disability.
The following drugs may help reduce symptoms:
• For spasticity, baclofen
• For cramps, quinine or phenytoin
• To decrease saliva production, a strong anticholinergic drug (eg, glycopyrrolate, amitriptyline,
benztropine, trihexyphenidyl, transdermal hyoscine, atropine)
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 185. Peripheral Nervous System & Motor Unit Disorders
1959

metadata: {'page': 1969, 'source': 'Merck Manual 19th ed.'}


**Observation — chunking.** The manual splits into roughly 17,000 chunks of about 870 characters on average (the mean is
below 1,000 because the splitter prefers to break at paragraph and sentence boundaries rather than mid-sentence). Each
page's running footer — edition, chapter title and page number — is retained deliberately: it lands inside the chunks
and gives the LLM and the reader an explicit chapter attribution at no extra cost.

### Embedding

**Embedding model — `all-MiniLM-L6-v2`.** A 6-layer, 384-dimension sentence-transformer that is the standard
choice for first-pass retrieval: it embeds about 1,000 chunks per second on a T4, its vectors are small (fast search,
small index), and it is trained on a billion sentence pairs, so it captures semantic similarity rather than keyword
overlap — "patchy hair loss / bald spots" lands near "alopecia areata" even though the words differ. Cosine
similarity is used as the distance metric because it is scale-invariant and is what the model was trained with.

In [25]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)

# Sanity check: embed a query and inspect the vector
sample_vector = embedding_model.embed_query("sudden patchy hair loss with bald spots on the scalp")
print(f"Embedding model     : {EMBEDDING_MODEL}")
print(f"Vector dimension    : {len(sample_vector)}")
print(f"First 8 components  : {np.round(sample_vector[:8], 4)}")

# Semantic similarity check: the query should be closer to the medical term than to an unrelated sentence
from numpy.linalg import norm
def cosine(a, b): return float(np.dot(a, b) / (norm(a) * norm(b)))
v_alopecia = embedding_model.embed_query("alopecia areata causes round bald patches on the scalp")
v_unrelated = embedding_model.embed_query("the femur is the longest bone in the human body")
print(f"\ncosine(query, 'alopecia areata ...')  = {cosine(sample_vector, v_alopecia):.3f}")
print(f"cosine(query, 'femur is the longest ...') = {cosine(sample_vector, v_unrelated):.3f}")

Embedding model     : all-MiniLM-L6-v2
Vector dimension    : 384
First 8 components  : [-0.039   0.0383  0.0407  0.0286 -0.03    0.0046  0.0114  0.0305]

cosine(query, 'alopecia areata ...')  = 0.775
cosine(query, 'femur is the longest ...') = -0.072


**Observation — embeddings.** The query about "patchy hair loss / bald spots" scores far higher against the sentence
that uses the clinical term *alopecia areata* than against an unrelated anatomical sentence, even though the query
and the alopecia sentence share few words. This is the property that lets a lay description in a question find the
correct chapter in the manual.

### Vector Database

**Vector store — ChromaDB.** An open-source, embedded vector database: no server to run, persists to local disk,
and supports cosine similarity search with metadata (page numbers) returned alongside every hit — which is what lets
the assistant cite the manual page it used. The collection is built once and persisted, so re-running the notebook
does not re-embed the corpus.

In [26]:
def build_vectorstore(docs, persist_directory, collection_name):
    """Embed a list of chunks into a persistent Chroma collection (reuses it if it already exists)."""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        store = Chroma(persist_directory=persist_directory, embedding_function=embedding_model,
                       collection_name=collection_name, collection_metadata={"hnsw:space": "cosine"})
        if store._collection.count() == len(docs):
            print(f"Reusing existing collection '{collection_name}' with {store._collection.count():,} vectors")
            return store
        del store
        shutil.rmtree(persist_directory)          # stale or partial index: rebuild from scratch
    start = time.time()
    store = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_name=collection_name,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"Embedded and indexed {store._collection.count():,} chunks in {time.time() - start:.1f} s")
    return store

vectorstore = build_vectorstore(chunks, "./merck_db_1000", "merck_1000_200")

Embedded and indexed 17,047 chunks in 75.2 s


### Retriever

**Retriever.** The retriever turns the vector store into a function *question → k most relevant chunks*. The starting
configuration is plain **similarity search with `k = 3`**: three chunks of roughly 250 tokens keep the prompt well inside the
context window while usually covering symptoms, diagnosis and treatment of one condition. Maximum-marginal-relevance
(MMR) search and larger `k` are compared in the fine-tuning section.

In [27]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Inspect what the retriever returns for one question
test_docs = retriever.invoke(questions["Q3"])
print("Question:", questions["Q3"], "\n")
for i, d in enumerate(test_docs, 1):
    print(f"--- Retrieved chunk {i} | PDF page {d.metadata['page']} ---")
    print(d.page_content[:500].strip(), "\n")

Question: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? 

--- Retrieved chunk 1 | PDF page 859 ---
hair loss associated with hyperandrogenemia.
Surgical options include follicle transplant, scalp flaps, and alopecia reduction. Few procedures have
been subjected to scientific scrutiny, but patients who are self-conscious about their hair loss may
consider them.
Hair loss due to other causes: Underlying disorders are treated.
Multiple treatment options for alopecia areata exist and include topical, intralesional, or, in severe cases,
systemic corticosteroids, topical minoxidil, topical anthrali 

--- Retrieved chunk 2 | PDF page 856 ---
alopecia occurs early in the course of the disease, and then permanent hair loss occurs as the disease
progresses. Scarring alopecias can be subdivided further into primary forms, where the target of
inflammation is

**Observation — retrieval check.** For the hair-loss question all three hits come from Chapter 86 *Hair Disorders*
(PDF pages 856, 858 and 859) even though the question never uses the word "alopecia": the embedding of "patchy hair
loss / bald spots" lands next to the alopecia section. Retrieval is therefore doing the diagnostic inference that
the raw LLM had to guess at in Section 1 — with a page citation. The chunk boundaries also show a limitation: the
alopecia-areata treatment list on page 859 begins at the very end of chunk 1 and is cut off, so with `k = 3` the
model may see the *causes* of hair loss but only the start of the *treatments*.

### System and User Prompt Template

The system message tells the model *how* to behave; the user template injects the retrieved context and the
question. Three design choices matter: (1) the model is told to answer **only from the context** and to say so when
the context is insufficient, which is the primary defence against hallucination; (2) it is asked for physician-level
language; (3) the context is placed *before* the question, which instruction-tuned models handle best.

In [28]:
qna_system_message = """You are a clinical decision-support assistant for physicians. You answer questions using ONLY the excerpts
from the Merck Manual of Diagnosis & Therapy that are provided as context.

Rules:
- Base every statement on the context. Do not add information from outside the context.
- If the context does not contain the information needed to answer, reply exactly: "The provided manual excerpts do not contain enough information to answer this question."
- Answer all parts of the question. Use clear clinical language, organised with short paragraphs or bullet points.
- Do not mention that you were given context or excerpts; simply answer."""

qna_user_message_template = """###Context
{context}

###Question
{question}"""

print(qna_system_message)
print("\n---\n")
print(qna_user_message_template)

You are a clinical decision-support assistant for physicians. You answer questions using ONLY the excerpts
from the Merck Manual of Diagnosis & Therapy that are provided as context.

Rules:
- Base every statement on the context. Do not add information from outside the context.
- If the context does not contain the information needed to answer, reply exactly: "The provided manual excerpts do not contain enough information to answer this question."
- Answer all parts of the question. Use clear clinical language, organised with short paragraphs or bullet points.
- Do not mention that you were given context or excerpts; simply answer.

---

###Context
{context}

###Question
{question}


### Response Function

In [29]:
def retrieve_context(user_input, retriever_obj):
    """Return the retrieved chunks and a single context string built from them."""
    relevant_document_chunks = retriever_obj.invoke(user_input)
    context_for_query = "\n\n".join(d.page_content for d in relevant_document_chunks)
    return relevant_document_chunks, context_for_query


def generate_rag_response(user_input, retriever_obj=None, k=3, max_tokens=256, temperature=0, top_p=0.95, top_k=50,
                          repeat_penalty=1.1, system_message=None, show_sources=True):
    """Retrieve context from the manual, build the prompt, and generate a grounded answer."""
    global qna_system_message, qna_user_message_template
    if retriever_obj is None:            # default: similarity search with the requested k on the main vector store
        retriever_obj = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    system_message = system_message or qna_system_message

    # 1. Retrieve relevant document chunks
    docs, context_for_query = retrieve_context(user_input, retriever_obj)

    # 2. Build the prompt: system message + context + question, in the Mistral chat template
    user_message = qna_user_message_template.replace("{context}", context_for_query).replace("{question}", user_input)
    prompt = build_prompt(user_message, system_message)

    # 3. Generate the response
    try:
        model_output = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repeat_penalty=repeat_penalty,
            stop=["[INST]", "</s>", "###Question"],
        )
        answer = model_output["choices"][0]["text"].strip()
    except Exception as e:
        answer = f"Sorry, I encountered the following error: \n {e}"

    if show_sources:
        pages_used = sorted({d.metadata["page"] for d in docs})
        print(f"[Retrieved {len(docs)} chunks from PDF pages {pages_used} | prompt ~{len(llm.tokenize(prompt.encode())):,} tokens]\n")
    return answer


def timed_rag_response(qid, **kwargs):
    start = time.time()
    answer = generate_rag_response(questions[qid], **kwargs)
    show(answer)
    print(f"\n[{len(answer.split())} words | {time.time() - start:.1f} s]")
    return answer

## Question Answering using RAG

The five questions are now answered with the baseline RAG configuration — **chunks 1,000/200, similarity search,
`k = 3`, greedy decoding, `max_tokens = 256`** — and the pages the answer was built from are printed above each
answer.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [30]:
results["RAG"]["Q1"] = timed_rag_response("Q1", k=3, max_tokens=BASELINE_MAX_TOKENS)

[Retrieved 3 chunks from PDF pages [2401, 2449, 2454] | prompt ~904 tokens]

The protocol for managing sepsis in a critical care unit involves providing supportive care, including keeping
the patient warm and controlling hemorrhage. First aid includes checking the airway and ventilation, avoiding
anything by mouth, and turning the patient's head to prevent aspiration if emesis occurs. Treatment begins
simultaneously with evaluation and includes providing supplemental oxygen and mechanical ventilation if
necessary. Two large IV catheters are inserted into separate peripheral veins, and a central venous line or
intraosseous needle may be used when peripheral veins cannot be promptly accessed. Aggressive fluid
resuscitation, antibiotics, surgical excision of infected or necrotic tissues, and drainage of pus are
essential components of treatment for severe sepsis and septic shock. Supportive care is also crucial to help
maintain dignity for dying patients in the ICU. (Refer to Chapters 222

**Observation — Q1 with RAG.** The three chunks come from PDF pages 2401, 2449 and 2454 — the opening of Chapter
227 *Sepsis & Septic Shock* plus two neighbouring pages from Chapter 222 (*Approach to the Critically Ill Patient*)
and Chapter 226 (*Shock*). The answer is therefore grounded but **contaminated by adjacent chapters**: it opens with
generic shock first aid (keep warm, control haemorrhage, two large-bore IV lines) and closes with a sentence about
maintaining dignity for dying ICU patients, neither of which is the sepsis protocol. The sepsis-specific treatment
text (fluids, broad-spectrum antibiotics, source control, glucose control, corticosteroids) is present but only as
the chapter's introductory summary. Provenance is now real — every sentence can be traced to a page — but `k = 3`
over a chapter *introduction* is not enough context for a protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [31]:
results["RAG"]["Q2"] = timed_rag_response("Q2", k=3, max_tokens=BASELINE_MAX_TOKENS)

[Retrieved 3 chunks from PDF pages [173, 174] | prompt ~857 tokens]

The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea,
vomiting, and anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with
cough and motion. Classical signs are right lower quadrant direct and rebound tenderness located at McBurney's
point. Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant.
If left untreated, appendicitis can lead to necrosis, gangrene, perforation, and the formation of an
appendiceal abscess.

Appendicitis cannot be cured via medicine alone. The standard treatment is surgical removal of the appendix.
This may involve an open appendectomy or a laparoscopic appendectomy. In cases where an abscess or
inflammatory mass has formed, the procedure may be limited to drainage of the abscess. Before the operation,
an NGT is inserted, and patients with signs of volume deplet

**Observation — Q2 with RAG.** Retrieval lands squarely on pages 173-174 of Chapter 11 *Acute Abdomen & Surgical
Gastroenterology*, and the answer is the manual's own text: periumbilical pain shifting to the right lower
quadrant, McBurney's point tenderness, the risk of necrosis, gangrene and perforation, then an explicit "cannot be
cured via medicine alone", open or laparoscopic appendectomy, drainage when an abscess has formed, and the
pre-operative steps (nasogastric tube, urine output, IV fluids and antibiotics). This is the question on which RAG
is unambiguously better than every prompted variant — more specific, fully attributable, and it answers the
medicine-versus-surgery part that the baseline never reached.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [32]:
results["RAG"]["Q3"] = timed_rag_response("Q3", k=3, max_tokens=BASELINE_MAX_TOKENS)

[Retrieved 3 chunks from PDF pages [856, 858, 859] | prompt ~1,007 tokens]

Sudden patchy hair loss, or localized bald spots on the scalp, can have various causes. The possible causes
include fungal infections (tinea capitis), bacterial infections, autoimmune disorders such as lupus
erythematosus, lichen planopilaris, and systemic sclerosis, traction alopecia, and less commonly,
trichotillomania.

For fungal infections, topical or oral antifungals are effective treatments. For autoimmune disorders,
underlying conditions are treated. Traction alopecia is addressed by eliminating physical stress to the scalp.
In cases of trichotillomania, behavior modification, clomipramine, or selective serotonin reuptake inhibitors
(SSRIs) may provide benefit.

To diagnose the cause of sudden patchy hair loss, fungal and bacterial cultures, immunofluorescence studies,
and daily hair counts can be helpful. Fungal and bacterial cultures may identify infections, while
immunofluorescence studies can help d

**Observation — Q3 with RAG.** The chunks come from the right chapter (pages 856, 858, 859 of *Hair Disorders*), yet
the answer never says "alopecia areata". It lists the manual's *differential* for patchy loss — tinea capitis,
bacterial infection, lupus, lichen planopilaris, traction alopecia, trichotillomania — with their treatments and
the diagnostic work-up (cultures, immunofluorescence, daily hair counts). The reason is visible in the retrieval
check above: the alopecia-areata treatment list begins at the very end of the page-859 chunk and is truncated, so
the model faithfully summarises what it was shown. Compared with the raw LLM, the RAG answer is less
"decisive" but entirely grounded — and the fine-tuning section shows that a larger `k`, MMR or smaller chunks all
recover the alopecia-areata treatment list.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [33]:
results["RAG"]["Q4"] = timed_rag_response("Q4", k=3, max_tokens=BASELINE_MAX_TOKENS)

[Retrieved 3 chunks from PDF pages [1834, 1844, 3648] | prompt ~589 tokens]

Treatments for a person with a physical injury to brain tissue include:

- Preventing systemic complications due to immobilization, such as pneumonia, urinary tract infections, and
thromboembolic disease.
- Providing good nutrition.
- Preventing pressure ulcers.
- Physical therapy to prevent limb contractures.
- Early intervention by rehabilitation specialists for maximal functional recovery.
- Prevention of secondary disabilities, including pressure ulcers and joint contractures.
- Prevention of pneumonia.
- Family education.
- Rehabilitation specialists should evaluate patients to establish baseline findings and prioritize treatment.
- Patients with severe cognitive dysfunction require extensive cognitive therapy, which is often begun
immediately after injury and continued for months or years.

Additionally, it's important to note that in some cases of irreversible or progressive disorders, such as
cancers i

**Observation — Q4 with RAG.** This is the notebook's clearest retrieval miss. The chunks come from pages 1834,
1844 and 3648 — general neurological rehabilitation and supportive-care text — and **not** from Chapter 324
*Traumatic Brain Injury* (pages 3405-3413). The answer is therefore grounded but off-target: pressure-ulcer
prevention, nutrition, contracture prevention, cognitive therapy, plus stray sentences about posterior-fossa
tumours and anticoagulation after stroke. Nothing about airway and oxygenation, intracranial-pressure control or
surgical evacuation. The question describes brain injury in lay terms ("physical injury to brain tissue ...
impairment of brain function"), and those words are semantically closer to the rehabilitation chapters than to
the TBI chapter's clinical vocabulary. Dense retrieval alone is vulnerable to this vocabulary gap.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [34]:
results["RAG"]["Q5"] = timed_rag_response("Q5", k=3, max_tokens=BASELINE_MAX_TOKENS)

[Retrieved 3 chunks from PDF pages [3387, 3389, 3512] | prompt ~741 tokens]

For a person with a fractured leg from a hiking injury, the following precautions and treatments are
recommended:

1. Rest: The injured limb should be rested to promote healing.
2. Ice and compression: Apply ice and compression using a thigh sleeve as soon as possible to reduce swelling
and pain.
3. Pain management: Nonsteroidal anti-inflammatory drugs (NSAIDs) and analgesics may be prescribed for pain
relief.
4. Crutches: Crutches might be necessary initially if walking is painful.
5. Wound care: Keep the wound clean and dry, apply nonadherent and impermeable dressings, and use antibiotic
ointment daily until the wound closure device is removed.
6. Early physician examination: For higher risk wounds or wounds in unreliable patients, early physician
examination is preferable.
7. Avoid weight-bearing activities: The person should avoid putting weight on the injured leg for several days
to promote better healing

**Observation — Q5 with RAG.** Retrieval spreads across pages 3387, 3389 (Chapter 323 *Fractures, Dislocations &
Sprains*) and 3512 (wound care). The answer is grounded but reads like soft-tissue-injury advice — rest, ice and
compression "using a thigh sleeve", NSAIDs, crutches, laceration dressing instructions — with fat embolism as the
only fracture-specific point. It does not mention splinting, reduction, casting or fixation, which the manual
covers a few pages further on. As with Q4, the scenario wording ("fractured their leg during a hiking trip") is far
from the manual's clinical headings, so the top-3 chunks are only partly the right ones.

### Fine-tuning

"Fine-tuning" here means tuning the **RAG pipeline** — chunking, retrieval and decoding — not the model weights.
Six configurations are compared on all five questions. Each changes one dimension relative to the baseline so that
its effect can be isolated.

| ID | Chunking | Retriever | LLM parameters | Hypothesis |
|---|---|---|---|---|
| **R-1** (baseline) | 1000 / 200 | similarity, k = 3 | temp 0, 256 tokens | Reference point |
| **R-2** | 1000 / 200 | similarity, **k = 5** | temp 0, 256 tokens | More context → more complete answers |
| **R-3** | 1000 / 200 | **MMR**, k = 5, fetch_k = 20, λ = 0.5 | temp 0, 256 tokens | Diverse chunks help scenario questions (Q5) |
| **R-4** | **500 / 100** | similarity, k = 6 | temp 0, 256 tokens | Finer chunks → more precise retrieval |
| **R-5** | 1000 / 200 | similarity, k = 3 | **temp 0.5, top_p 0.9**, 256 tokens | Does sampling hurt a grounded answer? |
| **R-6** | 1000 / 200 | similarity, k = 4 | temp 0, **512 tokens**, stricter system prompt | Longer budget + explicit refusal rule |

In [35]:
# A second, finer-grained index for configuration R-4 (500-character chunks with 100 overlap)
chunks_small = chunk_documents(cleaned_pages, 500, 100)
print(f"Number of chunks at 500/100 : {len(chunks_small):,}")
vectorstore_small = build_vectorstore(chunks_small, "./merck_db_500", "merck_500_100")

Number of chunks at 500/100 : 32,735
Embedded and indexed 32,735 chunks in 90.1 s


In [36]:
STRICT_SYSTEM = qna_system_message + """
- Structure the answer as: Diagnosis/condition, Immediate management, Definitive treatment, Follow-up and recovery.
- Prefer specific values (doses, targets, timings) from the context over general statements."""

rag_configs = {
    "R-1  baseline: 1000/200, similarity k=3":              dict(retriever_obj=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
                                                                 max_tokens=256, temperature=0.0, top_p=0.95),
    "R-2  1000/200, similarity k=5":                        dict(retriever_obj=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5}),
                                                                 max_tokens=256, temperature=0.0, top_p=0.95),
    "R-3  1000/200, MMR k=5 (fetch_k=20, lambda=0.5)":       dict(retriever_obj=vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}),
                                                                 max_tokens=256, temperature=0.0, top_p=0.95),
    "R-4  500/100, similarity k=6":                         dict(retriever_obj=vectorstore_small.as_retriever(search_type="similarity", search_kwargs={"k": 6}),
                                                                 max_tokens=256, temperature=0.0, top_p=0.95),
    "R-5  1000/200, similarity k=3, temp 0.5 / top_p 0.9": dict(retriever_obj=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
                                                                 max_tokens=256, temperature=0.5, top_p=0.90),
    "R-6  1000/200, similarity k=4, 512 tokens, strict":    dict(retriever_obj=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4}),
                                                                 max_tokens=512, temperature=0.0, top_p=0.95, system_message=STRICT_SYSTEM),
}

tuning_log = []          # one row per (configuration, question) for the summary table
rag_tuning_answers = {}  # full answers, keyed by configuration then question

for name, cfg in rag_configs.items():
    rag_tuning_answers[name] = {}
    for qid, query in questions.items():
        print("=" * 110)
        print(f"{name}   |   {qid}: {query[:80]}...")
        print("=" * 110)
        start = time.time()
        docs, _ = retrieve_context(query, cfg["retriever_obj"])
        answer = generate_rag_response(query, **cfg)
        show(answer)
        elapsed = time.time() - start
        print(f"\n[{len(answer.split())} words | {elapsed:.1f} s]\n")
        rag_tuning_answers[name][qid] = answer
        tuning_log.append({
            "config": name.split("  ")[0],
            "question": qid,
            "chunks": len(docs),
            "pages": ", ".join(str(p) for p in sorted({d.metadata["page"] for d in docs})),
            "words": len(answer.split()),
            "seconds": round(elapsed, 1),
            "refused": answer.startswith("The provided manual excerpts do not contain"),
        })

R-1  baseline: 1000/200, similarity k=3   |   Q1: What is the protocol for managing sepsis in a critical care unit?...
[Retrieved 3 chunks from PDF pages [2401, 2449, 2454] | prompt ~904 tokens]

The protocol for managing sepsis in a critical care unit involves providing supportive care, including keeping
the patient warm and controlling hemorrhage. First aid includes checking the airway and ventilation, avoiding
food and turning the head to prevent aspiration if emesis occurs. Treatment begins simultaneously with
evaluation and includes providing supplemental oxygen and mechanical ventilation if necessary. Two large IV
catheters are inserted into separate peripheral veins, and a central venous line or intraosseous needle may be
used when prompt access to peripheral veins is not possible. Aggressive fluid resuscitation, antibiotics,
surgical excision of infected or necrotic tissues, and drainage of pus are essential components of treatment
for severe sepsis and septic shock. Supportive

In [37]:
tuning_df = pd.DataFrame(tuning_log)
summary = tuning_df.pivot_table(index="config", values=["words", "seconds"], aggfunc="mean").round(1)
summary["refusals"] = tuning_df.groupby("config")["refused"].sum()
summary["avg_chunks"] = tuning_df.groupby("config")["chunks"].mean()
print("Average answer length, latency and refusals per configuration (over the 5 questions):\n")
display(summary)
print("\nPages retrieved per configuration and question:\n")
display(tuning_df.pivot(index="question", columns="config", values="pages"))

Average answer length, latency and refusals per configuration (over the 5 questions):



,seconds,words,refusals,avg_chunks
config,,,,
R-1,29.3,151.4,0,3.0
R-2,39.4,150.2,0,5.0
R-3,38.0,140.0,0,5.0
R-4,29.4,145.4,0,6.0
R-5,28.1,145.4,0,3.0
R-6,39.1,158.6,0,4.0



Pages retrieved per configuration and question:



config,R-1,R-2,R-3,R-4,R-5,R-6
question,,,,,,
Q1,"2401, 2449, 2454","2401, 2449, 2454","1083, 1448, 2401, 2454, 3673","2401, 2449, 2455, 2456, 2457","2401, 2449, 2454","2401, 2449, 2454"
Q2,"173, 174","173, 174, 3568","173, 174, 828, 2528, 3568","173, 174, 175, 3568","173, 174","173, 174, 3568"
Q3,"856, 858, 859","856, 858, 859","856, 857, 858, 859","856, 857, 859","856, 858, 859","856, 858, 859"
Q4,"1834, 1844, 3648","1834, 1843, 1844, 3413, 3648","1824, 1844, 3412, 3413, 3417","1821, 3406, 3407, 3413, 3444, 3648","1834, 1844, 3648","1834, 1844, 3413, 3648"
Q5,"3387, 3389, 3512","3387, 3389, 3391, 3512, 3647","2385, 3331, 3391, 3512, 3647","2385, 3314, 3387, 3391, 3397, 3512","3387, 3389, 3512","3387, 3389, 3391, 3512"


**Observations — RAG fine-tuning.** The pages table above is the most informative output in the notebook: it
shows *which* part of the manual each configuration put in front of the model.

* **`k = 5` (R-2).** For Q4 the two extra chunks finally include page 3413 from the TBI chapter, and for Q3 the
  answer now names alopecia areata and its immunotherapy options. For Q1 and Q2 the extra chunks are adjacent
  pages of the same section, so the answers change little. Latency rises from about 30 s to about 39 s per answer.
* **MMR (R-3).** Diversity cuts both ways. For Q3 it pulls in page 857 and yields the manual's complete
  alopecia-areata treatment list (topical, intralesional and systemic corticosteroids, minoxidil, anthralin,
  topical immunotherapy, PUVA); for Q4 it reaches pages 3412-3417 of the TBI chapter and surfaces anticonvulsants
  for seizures and the note that neuroprotective agents remain investigational. But for Q1 it drags in pages 1083,
  1448 and 3673 (sickle-cell anaemia, *C. difficile* infection control and *The Dying Patient*), and the sepsis
  answer duly wanders into aplastic crisis, splenectomy and end-of-life care; for Q5 it adds pages 2385 and 3331
  (prostheses and DVT prophylaxis). MMR helps when the question is under-specified and hurts when one section
  already contains the whole answer.
* **Smaller chunks (R-4).** The 500-character index (32,735 chunks) gives the most *specific* retrieval of all
  configurations: for Q1 it reaches pages 2455-2457, the actual sepsis treatment section (0.9 % saline, empiric
  broad-spectrum antibiotics adjusted to cultures, drainage of abscesses, glucose normalisation, replacement-dose
  corticosteroids, and the monitoring list); for Q4 it reaches pages 3406-3407 and 3413 of the TBI chapter; for
  Q5 it retrieves "immediate splinting followed by operative fixation for femoral shaft fractures". The cost is
  that a 500-character passage often ends mid-list, and one wrong chunk (lightning injury on page 3444 for Q4)
  can slip into the answer. Finer chunks with a re-ranker would be the natural next step.
* **Sampling (R-5).** With grounded context, temperature 0.5 mostly changes phrasing rather than facts because the
  context anchors the model — but it reintroduces run-to-run variation with no benefit, and for Q3 it led the
  answer toward androgenetic alopecia, which does not fit "sudden patchy" loss. Greedy decoding stays the choice.
* **Longer budget + strict structure (R-6).** Removing the 256-token cap and mandating *Diagnosis → Immediate
  management → Definitive treatment → Follow-up* produced the most readable and complete answers (Q5 now includes
  splinting/casting and surgical criteria; Q3 gives per-cause treatments including named corticosteroids), at a
  latency of about 39 s, the same as R-2. Its explicit refusal instruction was never triggered — the manual covered every
  question — which is what a zero refusal count in the summary table means.

**Selected configuration for evaluation:** 1,000/200 chunks, similarity search with `k = 4`, greedy decoding,
`max_tokens = 512`, and the strict structured system prompt (R-6) — the best balance of completeness, determinism
and readability. The retrieval evidence above also points to the first improvement a production system should make:
combine R-6's prompt with R-4's finer chunks (or a hybrid keyword + dense retriever), because for Q1, Q4 and Q5 the
finer index reached the specific treatment pages that the 1,000-character index missed.

In [38]:
# Final configuration used for the evaluation section
FINAL_K, FINAL_MAX_TOKENS = 4, 512
final_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": FINAL_K})
qna_system_message = STRICT_SYSTEM
print("Final RAG configuration: chunks 1000/200 | similarity search, k =", FINAL_K, "| temperature 0 | max_tokens", FINAL_MAX_TOKENS)

Final RAG configuration: chunks 1000/200 | similarity search, k = 4 | temperature 0 | max_tokens 512


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the questions from the previous section.

- We are using the same Mistral model for evaluation, so the LLM is essentially rating itself on how well it has performed the task. Two independent judge prompts are used:
  * **Groundedness** — is every claim in the answer supported by the retrieved context? (guards against hallucination)
  * **Relevance** — does the answer actually address what was asked? (guards against on-topic but unhelpful answers)
- Each judge returns a score from 1 to 5 with a short justification, so the scores can be tabulated and averaged.

In [39]:
groundedness_rater_system_message = """You are an impartial evaluator of a medical question-answering system.
You will be given a CONTEXT (excerpts from a medical manual), a QUESTION, and an ANSWER produced by an AI assistant.

Your task is to rate the GROUNDEDNESS of the ANSWER: the degree to which every claim in the ANSWER is directly
supported by the CONTEXT. Ignore whether the answer is helpful or complete; judge only whether it is supported.

Scoring scale:
5 - Every claim in the answer is explicitly supported by the context. No unsupported additions.
4 - Almost all claims are supported; at most one minor detail is not traceable to the context.
3 - The main points are supported but several details are not found in the context.
2 - Only a small part of the answer is supported; most content is not in the context.
1 - The answer is not supported by the context or contradicts it.

Respond in exactly this format:
Score: <integer 1-5>
Justification: <two or three sentences citing which claims are or are not supported>"""

In [40]:
relevance_rater_system_message = """You are an impartial evaluator of a medical question-answering system.
You will be given a CONTEXT (excerpts from a medical manual), a QUESTION, and an ANSWER produced by an AI assistant.

Your task is to rate the RELEVANCE of the ANSWER: the degree to which it directly and completely addresses every part
of the QUESTION as a physician would need. Ignore whether the claims are supported by the context; judge only whether
the answer is on-target and complete.

Scoring scale:
5 - Fully answers every part of the question, precisely and without irrelevant material.
4 - Answers the question well; one part is thin or a little irrelevant material is included.
3 - Addresses the main question but misses or only partially covers one or more sub-questions.
2 - Only loosely related to the question; most of what was asked is not answered.
1 - Does not address the question, or refuses although the context contained the answer.

Respond in exactly this format:
Score: <integer 1-5>
Justification: <two or three sentences explaining which parts of the question are or are not addressed>"""

In [41]:
user_message_template = """###Context
{context}

###Question
{question}

###Answer
{answer}"""

In [42]:
def parse_score(text):
    """Extract the integer score from a judge response of the form 'Score: <n>'."""
    match = re.search(r"Score:\s*\**\s*([1-5])", text)
    return int(match.group(1)) if match else None


def generate_ground_relevance_response(user_input, k=FINAL_K, max_tokens=FINAL_MAX_TOKENS, temperature=0, top_p=0.95, top_k=50):
    """Answer a question with the final RAG configuration, then have the LLM judge groundedness and relevance."""
    global qna_system_message, qna_user_message_template

    # 1. Retrieve context with the final retriever
    retriever_obj = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    docs, context_for_query = retrieve_context(user_input, retriever_obj)

    # 2. Generate the answer
    qna_user_message = qna_user_message_template.replace("{context}", context_for_query).replace("{question}", user_input)
    prompt = build_prompt(qna_user_message, qna_system_message)
    answer = llm(prompt=prompt, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k,
                 repeat_penalty=1.1, stop=["[INST]", "</s>", "###Question"])["choices"][0]["text"].strip()

    # 3. Judge groundedness and relevance (greedy decoding so the scores are reproducible)
    eval_user_message = user_message_template.format(context=context_for_query, question=user_input, answer=answer)
    groundedness_prompt = build_prompt(eval_user_message, groundedness_rater_system_message)
    relevance_prompt = build_prompt(eval_user_message, relevance_rater_system_message)

    response_1 = llm(prompt=groundedness_prompt, max_tokens=200, temperature=0, top_p=top_p, top_k=top_k, stop=["[INST]", "</s>"])
    response_2 = llm(prompt=relevance_prompt, max_tokens=200, temperature=0, top_p=top_p, top_k=top_k, stop=["[INST]", "</s>"])

    groundedness = response_1["choices"][0]["text"].strip()
    relevance = response_2["choices"][0]["text"].strip()
    return answer, groundedness, relevance, sorted({d.metadata["page"] for d in docs})


evaluation_log = []

def evaluate_question(qid):
    """Run the answer + two judges for one question, print everything, and log the scores."""
    start = time.time()
    answer, groundedness, relevance, pages_used = generate_ground_relevance_response(questions[qid])
    print(f"[Context from PDF pages {pages_used}]\n")
    print("ANSWER\n" + "-" * 110); show(answer)
    print("\nGROUNDEDNESS EVALUATION\n" + "-" * 110); show(groundedness)
    print("\nRELEVANCE EVALUATION\n" + "-" * 110); show(relevance)
    print(f"\n[{time.time() - start:.1f} s]")
    evaluation_log.append({"question": qid, "groundedness": parse_score(groundedness), "relevance": parse_score(relevance),
                           "answer_words": len(answer.split()), "pages": ", ".join(map(str, pages_used))})

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [43]:
evaluate_question("Q1")

[Context from PDF pages [2401, 2449, 2454]]

ANSWER
--------------------------------------------------------------------------------------------------------------
Diagnosis/condition: Sepsis in a critically ill patient

Immediate management:
- Keep the patient warm
- Control hemorrhage, if present
- Check and secure airway and ventilation
- Provide supplemental oxygen via face mask
- Insert two large IV catheters into separate peripheral veins
- Consider central venous line or intraosseous needle for access, especially in children
- Nothing by mouth
- Turn patient's head to avoid aspiration if emesis occurs

Definitive treatment:
- Aggressive fluid resuscitation
- Antibiotics based on culture and sensitivity results
- Surgical excision of infected or necrotic tissues and drainage of pus
- Supportive care, including mechanical ventilation if necessary
- Intensive control of blood glucose (in severe sepsis and septic shock)
- Administration of corticosteroids and activated protein C (in 

**Observation — Q1 evaluation.** Both judges award 5. The groundedness justification is fair — every bullet is lifted from the three chunks. The relevance score is generous: the answer still carries the generic shock first-aid steps and the end-of-life sentence from neighbouring chapters, which a human reviewer would mark as off-protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [44]:
evaluate_question("Q2")

[Context from PDF pages [173, 174, 3568]]

ANSWER
--------------------------------------------------------------------------------------------------------------
Appendicitis is characterized by epigastric or periumbilical pain followed by brief nausea, vomiting, and
anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and
motion. Classical signs include direct and rebound tenderness at McBurney's point. If left untreated,
appendicitis can lead to necrosis, gangrene, perforation, and the formation of an appendiceal abscess.

Appendicitis cannot be cured via medicine alone; surgical removal is the definitive treatment. The procedure
typically involves an open or laparoscopic appendectomy. In cases where an abscess or inflammatory mass has
formed, the surgery may be limited to drainage of the abscess.

Before the operation, an NGT is inserted, and patients with signs of volume depletion should have urine output
monitored with a catheter. Flui

**Observation — Q2 evaluation.** Both judges award 5, and here the scores are deserved: symptoms, the medicine-versus-surgery verdict, the procedure (open or laparoscopic appendectomy, drainage for abscess) and the pre-operative steps are all present and all traceable to pages 173-174.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [45]:
evaluate_question("Q3")

[Context from PDF pages [856, 858, 859]]

ANSWER
--------------------------------------------------------------------------------------------------------------
Diagnosis/condition: Alopecia presenting as sudden patchy hair loss can be caused by various conditions such
as alopecia areata, tinea capitis, trichotillomania, or scarring alopecia.

Immediate management:
- For suspected tinea capitis, initiate antifungal treatment.
- For trichotillomania, consider behavior modification techniques and/or pharmacotherapy with clomipramine or
an SSRI.

Definitive treatment:
- Alopecia areata: Topical corticosteroids (intralesional or lotions), topical immunotherapies (diphencyprone,
squaric acid dibutylester), systemic corticosteroids, or minoxidil may be considered.
- Tinea capitis: Oral antifungals are the primary treatment.
- Trichotillomania: Long-term management includes behavior modification techniques and pharmacotherapy with
clomipramine or an SSRI.
- Scarring alopecia: Treat underlying 

**Observation — Q3 evaluation.** Both judges award 5. With `k = 4` the answer now names alopecia areata and lists its treatments alongside the other causes of patchy loss, so relevance is legitimately high; groundedness is high because the answer is a structured paraphrase of the chapter.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [46]:
evaluate_question("Q4")

[Context from PDF pages [1834, 1844, 3413, 3648]]

ANSWER
--------------------------------------------------------------------------------------------------------------
Diagnosis/condition: Physical injury to brain tissue leading to temporary or permanent impairment of brain
function (Brain injury)

Immediate management:
- Preventing systemic complications due to immobilization (eg, pneumonia, UTI, thromboembolic disease)
- Providing good nutrition
- Preventing pressure ulcers
- Providing physical therapy to prevent limb contractures

Definitive treatment:
- There is no specific treatment for brain injury.

Follow-up and recovery:
- Early intervention by rehabilitation specialists is essential for maximal functional recovery.
- Rehabilitation services should be planned early.
- Patients with severe cognitive dysfunction require extensive cognitive therapy, often begun immediately
after injury and continued for months or years.
- Rehabilitation specialists should evaluate patients to es

**Observation — Q4 evaluation.** Both judges award 5, and this is where the method shows its limits. The answer is grounded — it repeats the rehabilitation text it was given — but "there is no specific treatment for brain injury" under *Definitive treatment* omits the acute management that Chapter 324 describes. The relevance judge sees the same four chunks the answerer saw and cannot know what was *not* retrieved; a retrieval miss is invisible to a context-based judge.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [47]:
evaluate_question("Q5")

[Context from PDF pages [3387, 3389, 3391, 3512]]

ANSWER
--------------------------------------------------------------------------------------------------------------
Diagnosis/condition: Fracture of the leg

Immediate management:
- Rest: The person should avoid putting weight on the injured leg.
- Ice and compression: Apply an ice pack to the affected area and use a thigh sleeve for compression.
- Pain relief: Prescribe NSAIDs and analgesics as necessary.
- Mobility aid: Provide crutches if walking is painful.

Definitive treatment:
- Splinting or casting: Immobilize the leg to prevent further injury and promote healing.
- Surgery (for severe cases): Consider surgical intervention for displaced fractures or those that affect the
vascular supply.

Follow-up and recovery:
- Wound care: Keep the wound clean and dry, apply nonadherent and impermeable dressings, and use antibiotic
ointment daily.
- Rehabilitation: Begin stretching exercises followed by strengthening exercises once the fr

**Observation — Q5 evaluation.** Both judges award 5. The answer is complete and well structured, but the groundedness judge accepts "tetanus prophylaxis" and "ice for 15-20 minutes every hour" as supported, which suggests it is checking topical overlap rather than claim-by-claim support. Groundedness scoring needs a stricter rubric or a stronger judge model to be trusted as a hallucination gate.

### Evaluation summary

In [48]:
evaluation_df = pd.DataFrame(evaluation_log).set_index("question")
display(evaluation_df)
print(f"\nMean groundedness : {evaluation_df['groundedness'].mean():.2f} / 5")
print(f"Mean relevance    : {evaluation_df['relevance'].mean():.2f} / 5")

,groundedness,relevance,answer_words,pages
question,,,,
Q1,5,5,164,"2401, 2449, 2454"
Q2,5,5,150,"173, 174, 3568"
Q3,5,5,147,"856, 858, 859"
Q4,5,5,124,"1834, 1844, 3413, 3648"
Q5,5,5,179,"3387, 3389, 3391, 3512"



Mean groundedness : 5.00 / 5
Mean relevance    : 5.00 / 5


**Observation — evaluation summary.** All five answers score 5/5 on both groundedness and relevance, so the final
configuration passes the assignment's evaluation cleanly — the answers are built from the retrieved manual text and
address the questions asked. The scores should nevertheless be read as an upper bound rather than clinical
validation, for three reasons that the per-question justifications make visible. First, the same Mistral model is
grading its own output, and it is lenient: it accepted a few details (Q5) that the context does not state verbatim,
and it did not penalise the adjacent-chapter material in Q1. Second, a context-based judge cannot detect a
retrieval miss: Q4's answer is faithful to the rehabilitation pages it was given and is therefore "grounded" and
"relevant", although the acute TBI management in Chapter 324 is absent. Third, a 1-5 scale from a single judge has
no resolution between "good" and "excellent". In production the same two prompts should be run by an independent,
stronger judge model, complemented by a retrieval-recall check against expected pages and by clinician review on a
sampled basis.

## Actionable Insights and Business Recommendations

### What the prototype demonstrates

**1. A raw LLM is fluent but unaccountable — and sometimes wrong.** Mistral-7B answered every clinical question
confidently from memory, but four of five baseline answers were truncated, one never reached the treatment part of
the question, and nothing it said could be traced to a source. Prompting made the answers complete and readable,
yet the "safe, constrained" configuration still recommended steroids for swelling after brain injury — a treatment
current trauma guidance does not support — and quoted a drug dose from memory, with no signal that anything was
wrong. Fluency without provenance is the
single biggest risk of putting a general-purpose LLM in front of clinicians.

**2. Prompt engineering controls form, not knowledge.** Structure and token budget (PE-3, PE-5) were the levers that
fixed completeness; a persona changed tone; chain-of-thought helped when the condition had to be inferred. None of
them corrected a fact the model had learned wrongly (minoxidil as an "immunomodulator" recurred in four answers) or
made it cite the manual, and a prompt that asks for determinism cannot deliver it — only `temperature = 0` does.

**3. Retrieval-augmented generation closes the provenance gap.** Grounding the same model in the Merck Manual gave
answers built from identifiable pages — Chapter 11 for appendicitis, Chapter 86 for alopecia, Chapter 227 for
sepsis, Chapter 323 for fractures — and retrieval, not the model, mapped the lay description "patchy hair loss" to the
alopecia section. Where RAG answers were weak, the weakness was *visible* in the retrieved page numbers, which is
something a raw LLM can never offer.

**4. Retrieval quality, not model quality, is the limiting factor.** The pages table in the fine-tuning section
shows that `k = 3` over 1,000-character chunks pulled in adjacent-chapter noise for sepsis and missed the TBI chapter
entirely for the brain-injury question, because the question's lay wording sits far from the chapter's clinical
vocabulary. Finer chunks (R-4) and a larger `k` recovered the correct treatment sections at negligible cost, and the
strict structured prompt (R-6) produced the most usable answers. Sampling temperature added nothing.

**5. Quality can be measured — but self-grading is lenient.** The groundedness and relevance judges scored the final
configuration 5/5 on every question, and their justifications point directly at what they can and cannot see: they
verify that an answer follows the retrieved text, but they cannot detect that the *wrong* text was retrieved (Q4),
and they let a few unsupported details pass (Q5). Evaluation must therefore combine an LLM judge with a
retrieval-recall check and periodic clinician review.

### Business recommendations

| Area | Recommendation | Expected benefit |
|---|---|---|
| **Deployment scope** | Launch as a *reference assistant* that returns a grounded answer **with page citations**, not as an autonomous diagnostic tool. Position it beside the EHR for clinicians, residents and triage nurses. | Faster access to trusted guidance without changing clinical accountability |
| **Knowledge base** | Extend the index beyond the Merck Manual to the hospital's own protocols (sepsis bundle, trauma pathway, formulary). Re-index on every protocol revision and stamp each chunk with a version and date. | Answers reflect *this* institution's standards, and stay current |
| **Safety controls** | Keep `temperature = 0`, keep the refusal rule, log every question, retrieved pages and answer, and route low groundedness scores to clinical review. | Deterministic, auditable behaviour — the requirement for any clinical governance committee |
| **Evaluation programme** | Build a gold set of 100-200 physician-written questions with reference answers; run the groundedness/relevance judges with an independent stronger model plus periodic clinician review. | Objective release criteria and regression detection when the model, index or prompts change |
| **Infrastructure** | Serve the open-weights model on-premise or in a private VPC; a single T4-class GPU indexed the 4,000-page manual in about a minute and answered in 20-50 seconds in this prototype; a production-grade GPU would bring that under 10 seconds. | No patient data or clinical queries leave the hospital's control; predictable cost |
| **Next technical iteration** | Hybrid retrieval (BM25 + dense), a cross-encoder re-ranker, section-aware chunking using the manual's chapter structure, and a larger embedding model. | Better coverage for scenario-style questions such as Q5, where dense retrieval alone under-performed |

### Expected impact

For a hospital, the value is measured in time and consistency: a resident who currently spends several minutes
locating a protocol in a 4,000-page manual gets a cited, structured answer in seconds; two clinicians asking the same
question receive the same answer; and every answer carries the page that justifies it. The prototype shows this is
achievable with open-weights components on a single GPU — the remaining work is governance, coverage and continuous
evaluation rather than new technology.

### Limitations of this study

* The Merck Manual edition used is a static reference; clinical guidance changes and the index must be refreshed.
* The LLM judged its own answers; absolute scores should be read as relative, not clinical validation.
* Five questions is a functional demonstration, not a statistically powered evaluation.
* Retrieval was purely dense (embedding-based); keyword-heavy queries such as drug names would benefit from hybrid search.

<font size=6 color='blue'>Power Ahead</font>
___